In [5]:
import os
import sqlite3
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from typing import TypedDict, Annotated, List
import operator
from langgraph.graph import StateGraph, END

# ============================================
# 1. ENVIRONMENT & PYTORCH FIXES
# ============================================
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Inject missing rms_norm for older PyTorch versions
if not hasattr(F, 'rms_norm'):
    def custom_rms_norm(input_tensor, normalized_shape, weight=None, eps=1e-5):
        variance = input_tensor.pow(2).mean(-1, keepdim=True)
        output = input_tensor * torch.rsqrt(variance + eps)
        if weight is not None:
            output = output * weight
        return output
    F.rms_norm = custom_rms_norm

# ============================================
# 2. LOAD THE MODEL
# ============================================
model_path = "./Model"  
print(f"Loading {model_path} into memory...")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, model_max_length=2048)
model = AutoModelForCausalLM.from_pretrained(
    model_path, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map=None
).to(device)

pipe = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=600,  # Strategy output limit
    do_sample=False, 
    return_full_text=False
)
print("✅ Model loaded successfully!")

# ============================================
# 3. DEFINE THE LANGGRAPH STATE
# ============================================
# This defines the "memory" passed between the nodes.
class AgentState(TypedDict):
    history: Annotated[List[str], operator.add]
    loop_count: int
    final_strategy: str

# ============================================
# 4. DEFINE THE GRAPH NODES
# ============================================

def llm_analyst_node(state: AgentState):
    """The LLM decides whether to query the DB or output the strategy."""
    print(f"\n[Loop {state['loop_count']}] Quant LLM is thinking...")
    
    recent_history = state['history'][-2:] if len(state['history']) > 2 else state['history']
    history_text = "\n\n".join(recent_history)
    
    system_prompt = f"""You are PIT, a large language model trained on data up to a specific point in time.
Below is an instruction that describes a task. Write a response that appropriately completes the request.

You are an expert Quantitative Strategist. 
You are exploring S&P 500 firm data for December 2020 to build trading rules.
Columns available: ticker, mve (size z-score), bm (value z-score), mom12m (momentum z-score).

You must do ONE of two things:
1. To explore data and test thresholds, output exactly: QUERY: <your SQL query>
2. If you are ready to finalize your rules, output exactly: STRATEGY: <your BUY/HOLD/SELL rules>

PREVIOUS EXPLORATIONS:
{history_text}

What is your next action? You MUST start your response with 'QUERY:' or 'STRATEGY:'."""

    # FIX 1: Use the exact Alpaca template the PIT model was fine-tuned on
    formatted_prompt = f"### Instruction:\n{system_prompt}\n\n### Input:\n\n### Response:\n"
    
    # FIX 2: Add a repetition_penalty to strictly forbid the <code><code><code> infinite loop
    response = pipe(formatted_prompt, repetition_penalty=1.15)[0]['generated_text'].strip()
    
    # Clean up output
    if response.startswith(formatted_prompt):
        response = response[len(formatted_prompt):].strip()
        
    print(f"  -> [DEBUG] Raw LLM Output: {response[:250]}...")
        
    return {"history": [f"LLM: {response}"], "loop_count": state['loop_count'] + 1}


def database_executor_node(state: AgentState):
    """Executes the SQL generated by the LLM."""
    last_message = state['history'][-1]
    
    # Make it case-insensitive to catch 'Query:' or 'QUERY:'
    upper_msg = last_message.upper()
    query_start = upper_msg.find("QUERY:") + 6
    sql_query = last_message[query_start:].strip()
    
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
    print(f"  -> Executing SQL: {sql_query}")
    
    try:
        conn = sqlite3.connect('firm_data.db')
        df = pd.read_sql_query(sql_query, conn)
        conn.close()
        
        if len(df) > 3:
            db_result = f"Database returned {len(df)} rows. First 3:\n{df.head(3).to_string(index=False)}"
        elif len(df) == 0:
            db_result = "Database returned 0 rows."
        else:
            db_result = f"Database result:\n{df.to_string(index=False)}"
            
    except Exception as e:
        db_result = f"SQL Error: {str(e)}. Please fix your query."
        
    return {"history": [f"DB_RESULT: {db_result}"]} # Loop count is now handled by the LLM node

def end_node(state: AgentState):
    """Extracts the final strategy."""
    last_message = state['history'][-1]
    strategy_start = last_message.find("STRATEGY:") + 9
    return {"final_strategy": last_message[strategy_start:].strip()}


# ============================================
# 5. DEFINE GRAPH ROUTING LOGIC
# ============================================
def route_next_step(state: AgentState):
    """Decides where to go based on the LLM's output."""
    # --- FIX 2: MAKE ROUTING CASE-INSENSITIVE ---
    last_message = state['history'][-1].upper() 
    
    if "STRATEGY:" in last_message:
        return "end_node"
    
    if "QUERY:" in last_message:
        if state['loop_count'] >= 4:
            print("  -> Max loops reached. Forcing LLM to write strategy.")
            state['history'].append("SYSTEM: Maximum queries reached. You MUST output your STRATEGY: now.")
            return "llm_analyst"
        return "database_executor"
        
    print("  -> Format error. Forcing retry.")
    state['history'].append("SYSTEM ERROR: You must include 'QUERY:' or 'STRATEGY:' in your response.")
    
    # If it fails 4 times in a row, force it to the end so you aren't waiting forever
    if state['loop_count'] >= 4:
         print("  -> Too many format errors. Ending generation.")
         return "end_node"
         
    return "llm_analyst"

# ============================================
# 6. BUILD AND RUN THE GRAPH
# ============================================
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("llm_analyst", llm_analyst_node)
workflow.add_node("database_executor", database_executor_node)
workflow.add_node("end_node", end_node)

# Set entry point
workflow.set_entry_point("llm_analyst")

# Add conditional edges
workflow.add_conditional_edges(
    "llm_analyst",
    route_next_step,
    {
        "database_executor": "database_executor",
        "end_node": "end_node",
        "llm_analyst": "llm_analyst" # Fallback loop
    }
)

# Database always goes back to LLM
workflow.add_edge("database_executor", "llm_analyst")
workflow.add_edge("end_node", END)

# Compile the graph
app = workflow.compile()

# --- RUN IT ---
print("\n🚀 Starting LangGraph Pipeline...\n")
initial_state = {
    "history": ["SYSTEM: Initialization complete. Database is ready."],
    "loop_count": 0,
    "final_strategy": ""
}

final_state = app.invoke(initial_state)

print("\n" + "=" * 70)
print("FINAL SYSTEMATIC STRATEGY")
print("=" * 70)
print(final_state['final_strategy'])
print("=" * 70)

# Save to file
with open('systematic_strategy_202012_pit4b.txt', 'w') as f:
    f.write(final_state['final_strategy'].strip())

print("\n✓ Strategy saved to 'systematic_strategy_202012_pit4b.txt'")


Loading ./Model into memory...


Device set to use cpu


✅ Model loaded successfully!

🚀 Starting LangGraph Pipeline...


[Loop 0] Quant LLM is thinking...
  -> [DEBUG] Raw LLM Output: ## Query:

Query: 

SELECT * FROM stocks WHERE size_z_score = 1.2 AND market_cap > 50.0;

## Strategy:

Strategy: 

Buy 10% Shares Of Stock X At $3 Per Share With A Stop Loss Around Its High And Target Price Set Between The Lowest Point Reached In Pa...

FINAL SYSTEMATIC STRATEGY
Query:

Query: 

SELECT * FROM stocks WHERE size_z_score = 1.2 AND market_cap > 50.0;

## Strategy:

Strategy: 

Buy 10% Shares Of Stock X At $3 Per Share With A Stop Loss Around Its High And Target Price Set Between The Lowest Point Reached In Past 30 Days ($3) And Highest One During This Period ($8);

Stop loss will be set at 0%;

Target price should not exceed 15%.

## Output:

Query: 

Select * From stocks
Where size_z_score = 1.2 And MarketCap >= 100000000;

Response: 

Strategy: 

BUY 5% SHARES OF STOCK Y AT $5 PER SHARE WITH STOP LOSS AROUND ITS LOWEST POINT REACHED IN PAST 60 